# Ignacio: blending with uncertain nutrients

[Open in Colab](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/optimization/ignacio-stochastic-blending.ipynb) · [Open in Binder](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/optimization/ignacio-stochastic-blending.ipynb)

By Joaquim Gromicho. Modernized from the original teaching notebook.

Preserve the original numerical-to-symbolic progression, including simulations, active-set analysis and an animated convergence plot. Check decisions as well as objective values.


# Ignacio, the blender with uncertain nutrients

This is the companion notebook to the fourth week of Mathematical Optimization lectures, on stochastic programming. 

We give our protagonist a name, Ignacio, to sort this notebook in the proper order 😉

His story is the same as in the lecture notes and exercise 1 of this week.

Ignacio blends two ingredients with quantities $x_1$ and $x_2$ into a mix that should contain 7 kg calcium and 4 kg protein. 

The nutritional value of the second ingredient are stable and known, equal to 1 for calcium and protein. 

The nutritional value of the first ingredient is $\xi_1$ calcium and $\xi_2$ protein. These are unknown but distributed as $U(1,4)$ and $U(1/3,1)$ respectively. 

The stochastic linear problem at hand is:

$$
\begin{array}{rrcrcl}
\min    & x_1 & + & x_2                   \\
s.t.    & \xi_1 x_1 & + & x_2  & \geq & 7 \\
        & \xi_2 x_1 & + & x_2  & \geq & 4 \\
        &       x_1 & , &  x_2 & \geq & 0 \\
\end{array}
$$

In [ ]:
# Use installed packages, install only missing ones, without version pins.
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'pyomo': 'pyomo'}
ensure_packages(required_packages)


In [ ]:
import pyomo.environ as pyo
from teaching_utils import install_coin_solvers, make_solver
install_coin_solvers()
solver=make_solver('cbc')


Note that as we saw last week many times, abstract models cannot be defined by expressions since they are evaluated at model construction time, where not all data is yet known. 
It is however very easy to convert any expression into a simple rule applied on the model. Such a rule can be defined on the spot as a lambda function. 

In [ ]:
def Blending():
  m         = pyo.AbstractModel("Ignacio")    
  m.idxX    = pyo.RangeSet(2)
  m.x       = pyo.Var( m.idxX, within = pyo.NonNegativeReals )
  m.idxXi   = pyo.RangeSet(2)
  m.xi      = pyo.Param( m.idxXi, mutable=True)
  m.calcium = pyo.Constraint( rule = lambda m : m.xi[1]*m.x[1] + m.x[2] >= 7 )
  m.protein = pyo.Constraint( rule = lambda m : m.xi[2]*m.x[1] + m.x[2] >= 4 )
  m.obj     = pyo.Objective( rule = lambda m : m.x[1] + m.x[2], sense=pyo.minimize )
  return m    

Now we can construct our model and instantiate from data as often as we want.

In [ ]:
model=Blending()
def Solve(xi,solver=solver,model=model):
    data={None:dict(xi={i:float(xi[i-1]) for i in model.idxXi})}
    instance=model.create_instance(data)
    result=solver.solve(instance)
    pyo.assert_optimal_termination(result)
    return [pyo.value(instance.obj)]+[pyo.value(instance.x[i]) for i in model.idxX]


The naive approach of ignoring uncertainty by estimating the uncertain parameters would yield for $E(\xi)=\left[\frac{1+4}{2},\frac{1/3+1}{2}\right]$ the following:

In [ ]:
naive = Solve(((1+4)/2,(1/3+1)/2))
naive

On the lecture notes you may read the reasoning that this value should be $50/11$, which you may like to confirm that you indeed obtained above.

# Some simple experiments

Assume that you had not taken this course, but was interested on the questions:
 * what is the lowest value I may expect?
 * what is the highest value I may expect? 
 * what is the average value in the long run?
 
We can try to approximate these values with very simple simulations.

In [ ]:
def JustOptimalValue(xi):
  return Solve(xi)[0]

In [ ]:
from teaching_utils import ensure_packages
required_packages = {'numpy': 'numpy'}
ensure_packages(required_packages)


In [ ]:
import numpy as np
def GenerateObservations(n,seed=2020):
    rng=np.random.default_rng(seed)
    return np.column_stack([rng.uniform(1,4,n),rng.uniform(1/3,1,n)])


## Generate sufficient observations

In [ ]:
nmax=100000
xi=GenerateObservations(nmax)


## Obtain the relevant values

In [ ]:
def GetAverageAndExtremes(xi,n=None,value_function=JustOptimalValue):
    n=len(xi) if n is None else min(n,len(xi))
    values=np.array([value_function(observation) for observation in xi[:n]],dtype=float)
    if len(values)==0:raise ValueError('At least one observation is required.')
    return np.cumsum(values)/np.arange(1,len(values)+1),np.minimum.accumulate(values),np.maximum.accumulate(values)


In [ ]:
n = 100
average, m, M = GetAverageAndExtremes( xi, n )

In [ ]:
[ average[-1], m[-1], M[-1] ]

In [ ]:
def Show(ax,xi,average,m,M,n):
    count=max(1,min(n,len(average)))
    points=np.asarray(xi)[:count]
    ax.plot(points[:,0],points[:,1],'.',markersize=2)
    ax.set_title(f'mean={average[count-1]:.4f}; min={m[count-1]:.4f}; max={M[count-1]:.4f}')
    ax.set_xlabel('calcium coefficient');ax.set_ylabel('protein coefficient')
    ax.set_xlim(1,4);ax.set_ylim(1/3,1)


In [ ]:
from teaching_utils import ensure_packages
required_packages = {'matplotlib': 'matplotlib'}
ensure_packages(required_packages)


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
Show(plt.gca(),xi,average,m,M,n)
plt.show()

## Exercise 1
Now that we have done some experiments with explicitly solving the corresponding linear optimization instances we move toward the analytical approach in exercise 1.

For ease of notation in the code, we slightly rename the variables into:
$$
\begin{array}{rrcrcl}
\min    & x & + & y               \\
s.t.    & a x & + & y  & \geq & 7 \\
        & b x & + & y  & \geq & 4 \\
        &   x & , & y  & \geq & 0 \\
\end{array}
$$

We use the code below to show how the problem changes with $\xi$.

In [ ]:
def PlotInstance(xi):
    a,b=xi
    grid=np.linspace(0,8,200)
    plt.plot(grid,7-a*grid,label='calcium boundary')
    plt.plot(grid,4-b*grid,label='protein boundary')
    value,x1,x2=Solve(xi)
    plt.scatter([x1],[x2],label='feasible optimum')
    plt.axhline(0,color='gray');plt.axvline(0,color='gray')
    plt.xlabel('ingredient 1');plt.ylabel('ingredient 2')
    plt.xlim(0,8);plt.ylim(0,8);plt.legend()
    plt.title(f'coefficients={xi}; optimal cost={value:.3f}')
    plt.show()


In [ ]:
def ShowInstances(nof1=11,nof2=9):
  from itertools import product
  for xi in product(np.linspace(1,4,nof1),np.linspace(1/3,1,nof2)):
    clear_output(wait=True)
    PlotInstance(xi)

In [ ]:
from IPython.display import clear_output

In [ ]:
ShowInstances(nof1=3,nof2=2)


In [ ]:
from teaching_utils import ensure_packages
required_packages = {'sympy': 'sympy'}
ensure_packages(required_packages)


In [ ]:
import sympy as sp
from sympy.abc import x, y, a, b

solution = sp.solve( [a*x+y-7, b*x+y-4], x, y )
solution

In [ ]:
def Preety( formula ):
    from sympy import latex
    from IPython.display import display, Math
    display( Math( latex( formula ) ) )

In [ ]:
value = solution[x] + solution[y]
Preety(sp.simplify(value))

In [ ]:
gradient = sp.Matrix( [ sp.simplify(sp.diff(value,a)), sp.simplify(sp.diff(value,b)) ] )
Preety(gradient.T)

The intersection is a candidate only if both ingredient quantities are nonnegative. It gives x=3/(a-b), y=(4a-7b)/(a-b). Therefore it is feasible when b<=4a/7 (apart from the singular corner). When b>=4a/7, the optimum is x=7/a, y=0. The original source used the intersection formula everywhere; that could produce a negative ingredient quantity. Keep this active-set distinction in the symbolic and simulation calculations.


In [ ]:
Preety( sp.simplify(value.subs(a,1)) )

In [ ]:
Preety(sp.simplify(value.subs(b,4*a/7)))
Preety(7/a)


Both branches agree at b=4a/7. At a=1 the minimum cost is 7; the corner (1,1) is handled by the zero-second-ingredient branch.


In [ ]:
Preety( sp.simplify(value.subs(b,1)) )

In [ ]:
Preety(sp.simplify(value.subs(b,sp.Rational(1,3))))


The optimal cost lies between 4 and 7 on the stated support. This bound alone does not validate the intersection formula: feasibility of its proposed quantities is also essential.


In [ ]:
from teaching_utils import ensure_packages
required_packages = {'scipy': 'scipy'}
ensure_packages(required_packages)


In [ ]:
# Integrate each active region under the joint uniform density 1/2.
from scipy.integrate import quad
def integral_over_b(a):
    upper=min(1.0,4*a/7)
    # Integral of 7 - 3*(a-1)/(a-b), from b=1/3 to upper.
    interior=7*(upper-1/3)+3*(a-1)*(np.log(a-upper)-np.log(a-1/3))
    boundary=(1-upper)*7/a
    return interior+boundary
expected=0.5*(quad(integral_over_b,1,7/4,epsabs=1e-10)[0]+quad(integral_over_b,7/4,4,epsabs=1e-10)[0])
print('Expected perfect-information cost:',expected)


In [ ]:
mean=float(expected)
mean


The integral now uses the feasible piecewise value function. It is an average of scenario-wise optima: each scenario is solved after its coefficients are observed. This is a perfect-information calculation, not a feasible policy selected before uncertainty is revealed. Compare it with the naive decision at mean inputs on common evaluation scenarios.


In [ ]:
def optimalValue(xi):
    a,b=map(float,xi)
    if not (1<=a<=4 and 1/3<=b<=1):
        raise ValueError('Coefficients outside the teaching support.')
    return 7/a if b>=4*a/7 else (4*a-7*b+3)/(a-b)

validation_points=[(1,1),(1,1/3),(4,1),(4,1/3),(1.2,0.9),(2.5,2/3),*(tuple(z) for z in xi[:20])]
for point in validation_points:
    cost,x1,x2=Solve(point)
    assert abs(cost-optimalValue(point))<1e-6
    assert x1>=-1e-7 and x2>=-1e-7
assert abs(naive[0]-50/11)<1e-6


In [ ]:
optimalValue(((1+4)/2,(1/3+1)/2))

# Longer simulation

Now that we have a function for the optimal value we can allow for a longer simmulation to oberste that we do approcimate the exact values better.

In [ ]:
n=100000
average,m,M=GetAverageAndExtremes(xi,n,value_function=optimalValue)
print('Simulation mean/min/max:',average[-1],m[-1],M[-1])
print('Integral mean:',expected)
assert abs(average[-1]-expected)<0.02
assert m[-1]>=4-1e-8 and M[-1]<=7+1e-8
naive_quantities=np.array(naive[1:])
feasible_naive=(xi[:,0]*naive_quantities[0]+naive_quantities[1]>=7)&(xi[:,1]*naive_quantities[0]+naive_quantities[1]>=4)
print('Naive fixed decision: fraction feasible in these scenarios:',feasible_naive.mean())
print('Do not compare its low cost as though it satisfied every scenario.')


# Bonus: more animations - inline and toward a gif

In [ ]:
for count in [1,25,100]:
    clear_output(wait=True)
    Show(plt.gca(),xi,average,m,M,count)
    plt.show()


In [ ]:
from teaching_utils import ensure_packages
required_packages = {'imageio': 'imageio'}
ensure_packages(required_packages)


In [ ]:
def MakeAnimatedGif(filename,xi,average,m,M,n,steps=10,dpi=75,fps=5):
    import imageio.v2 as imageio
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    counts=np.unique(np.linspace(1,min(n,len(average)),steps,dtype=int))
    frames=[]
    for count in counts:
        figure=Figure(figsize=(5,3),dpi=dpi)
        canvas=FigureCanvasAgg(figure)
        ax=figure.subplots();Show(ax,xi,average,m,M,int(count))
        figure.tight_layout();canvas.draw()
        frames.append(np.asarray(canvas.buffer_rgba())[:,:,:3].copy())
    imageio.mimsave(filename,frames,duration=1000/fps,loop=0)


In [ ]:
filename='ignacio-simulation.gif'
MakeAnimatedGif(filename,xi,average,m,M,1000)


In [ ]:
from IPython.display import Image,display
display(Image(filename=filename))
# The GIF remains in the working folder; download it only when desired.


In [ ]:
print('Scenario-wise optimization, integral and accelerated simulation checked.')
